In [1]:
from peft import PeftModel
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import torch
import numpy as np
import random
import os

In [2]:
# Set random states.
def set_random_states(random_state):
    # Set various random seeds.
    np.random.seed(random_state)
    random.seed(random_state)
    torch.manual_seed(random_state)
    torch.cuda.manual_seed_all(random_state)
    os.environ["PYTHONHASHSEED"] = str(random_state)
    os.environ["TOKENIZERS_PARALLELISM"] = "false"
    try:
        torch.use_deterministic_algorithms(True)
    except Exception:
        pass
    return random_state
RANDOM_STATE = set_random_states(1618)

# Make constant variables.
MODEL_NAME = "distilbert-base-uncased"

In [3]:
# Load fine-tuned model.
base_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

model = PeftModel.from_pretrained(base_model, "synth_lora_model")

tokenizer = AutoTokenizer.from_pretrained("synth_lora_model")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [4]:
# Run inference.
text = "Patient does not have access to food support"

inputs = tokenizer(text, return_tensors="pt")

outputs = model(**inputs)

pred = torch.argmax(outputs.logits)

labels = {0: "met", 1: "unmet"}

print(labels[pred.item()])

unmet


In [6]:
# Simple prediction function.
def predict(text):

    inputs = tokenizer(text, return_tensors="pt")

    with torch.no_grad():
        outputs = model(**inputs)

    pred = torch.argmax(outputs.logits).item()

    labels = {0: "met", 1: "unmet"}

    return labels[pred]

In [7]:
predict("Patient has no housing support")

'unmet'

In [8]:
predict("Patient has housing support")

'met'

In [9]:
predict("Patient had a pleasant day playing chess.")

'met'

In [10]:
predict("Patient had incontinence episode.")

'unmet'